# Clasificación: riesgo de cancelación de pedidos

Este notebook utiliza las colecciones `pedidos`, `usuarios` y `productos`. La variable objetivo es `clase_y`: **Cancelado = 1** y **Entregado = 0**. No modifica ni agrega campos a MongoDB; las transformaciones existen únicamente en el DataFrame.

## 1. Dependencias
Si hace falta, ejecuta primero: `pip install pymongo python-dotenv pandas scikit-learn matplotlib seaborn joblib`.

In [ ]:
import os
from pathlib import Path
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pymongo import MongoClient
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

env_path = next((p for p in [Path.cwd() / '.env', Path.cwd().parent / '.env'] if p.exists()), None)
load_dotenv(env_path)
MONGO_URI = os.getenv('MONGO_URI')
if not MONGO_URI:
    raise RuntimeError('No se encontró MONGO_URI. Abre Jupyter desde pryBinaBack o carga el archivo .env.')

## 2. Extracción y transformación del dataset

In [ ]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000)
db = client.get_default_database()
pedidos = list(db.pedidos.find(
    {'estado': {'$in': ['Entregado', 'Cancelado']}},
    {'usuario': 1, 'productos': 1, 'total': 1, 'metodoPago': 1, 'estado': 1, 'costoEnvio': 1, 'createdAt': 1}
))
len(pedidos)

In [ ]:
usuarios = {str(u['_id']): u for u in db.usuarios.find({}, {'nombre': 1, 'ap': 1, 'am': 1, 'fechaNacimiento': 1})}
def calcular_edad(fecha_nacimiento, fecha_referencia):
    if not fecha_nacimiento or not fecha_referencia: return None
    return fecha_referencia.year - fecha_nacimiento.year - ((fecha_referencia.month, fecha_referencia.day) < (fecha_nacimiento.month, fecha_nacimiento.day))

filas = []
historial = {}
for pedido in sorted(pedidos, key=lambda p: p.get('createdAt')):
    usuario_id = str(pedido.get('usuario', ''))
    usuario = usuarios.get(usuario_id, {})
    previos, cancelados = historial.get(usuario_id, (0, 0))
    porcentaje_cancelados_previos = (cancelados / previos * 100) if previos else 0
    for item in pedido.get('productos', []):
        filas.append({
            'pedido_id': str(pedido['_id']),
            'usuario': usuario_id,
            'nombre_usuario': ' '.join(filter(None, [usuario.get('nombre'), usuario.get('ap'), usuario.get('am')])),
            'edad': calcular_edad(usuario.get('fechaNacimiento'), pedido.get('createdAt')),
            'producto': str(item.get('producto', '')),
            'cantidad': item.get('cantidad', 0),
            'precio': item.get('precio', 0),
            'total': pedido.get('total', 0),
            'costo_envio': pedido.get('costoEnvio', 0),
            'metodo_pago': pedido.get('metodoPago', 'Sin definir'),
            'fecha': pedido.get('createdAt'),
            'porcentaje_cancelados_previos': porcentaje_cancelados_previos,
            'clase_y': 1 if pedido.get('estado') == 'Cancelado' else 0
        })
    historial[usuario_id] = (previos + 1, cancelados + (1 if pedido.get('estado') == 'Cancelado' else 0))

df = pd.DataFrame(filas)
print('Filas producto-pedido:', len(df))
print('Pedidos:', df['pedido_id'].nunique())
display(df.head())
display(df['clase_y'].value_counts().rename({0: 'Entregado', 1: 'Cancelado'}))

## 3. Separación sin fuga de información
Las líneas del mismo pedido permanecen juntas mediante `GroupShuffleSplit`; así un pedido no puede aparecer simultáneamente en entrenamiento y prueba.

In [ ]:
features = ['producto', 'cantidad', 'precio', 'total', 'costo_envio', 'metodo_pago', 'edad', 'porcentaje_cancelados_previos']
X = df[features]
y = df['clase_y']
grupos = df['pedido_id']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, grupos))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
grupos_test = grupos.iloc[test_idx]
print('Pedidos entrenamiento:', grupos.iloc[train_idx].nunique())
print('Pedidos prueba:', grupos_test.nunique())

In [ ]:
categoricas = ['producto', 'metodo_pago']
numericas = ['cantidad', 'precio', 'total', 'costo_envio', 'edad', 'porcentaje_cancelados_previos']
preprocesador = ColumnTransformer([
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categoricas),
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), numericas)
])
modelo = Pipeline([
    ('preprocesamiento', preprocesador),
    ('clasificador', KNeighborsClassifier(n_neighbors=15, weights='distance', metric='cosine'))
])
modelo.fit(X_train, y_train)

## 4. Evaluación por pedido

In [ ]:
prob_linea = modelo.predict_proba(X_test)[:, 1]
evaluacion = pd.DataFrame({'pedido_id': grupos_test.values, 'real': y_test.values, 'probabilidad': prob_linea})
evaluacion = evaluacion.groupby('pedido_id', as_index=False).agg(real=('real', 'first'), probabilidad=('probabilidad', 'mean'))
evaluacion['prediccion'] = (evaluacion['probabilidad'] >= 0.5).astype(int)
print(classification_report(evaluacion['real'], evaluacion['prediccion'], target_names=['Entregado', 'Cancelado'], zero_division=0))
if evaluacion['real'].nunique() == 2:
    print('ROC-AUC:', round(roc_auc_score(evaluacion['real'], evaluacion['probabilidad']), 3))
cm = confusion_matrix(evaluacion['real'], evaluacion['prediccion'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Entregado', 'Cancelado'], yticklabels=['Entregado', 'Cancelado'])
plt.xlabel('Predicción'); plt.ylabel('Real'); plt.title('Matriz de confusión'); plt.show()

## 5. Predecir pedidos pendientes o pagados

In [ ]:
pendientes = list(db.pedidos.find({'estado': {'$in': ['Pendiente', 'Pagado']}}))
filas_pendientes = []
for pedido in pendientes:
    for item in pedido.get('productos', []):
        filas_pendientes.append({
            'pedido_id': str(pedido['_id']), 'usuario': str(pedido.get('usuario', '')),
            'nombre_usuario': ' '.join(filter(None, [usuarios.get(str(pedido.get('usuario', '')), {}).get('nombre'), usuarios.get(str(pedido.get('usuario', '')), {}).get('ap'), usuarios.get(str(pedido.get('usuario', '')), {}).get('am')])),
            'edad': calcular_edad(usuarios.get(str(pedido.get('usuario', '')), {}).get('fechaNacimiento'), pedido.get('createdAt')),
            'producto': str(item.get('producto', '')), 'cantidad': item.get('cantidad', 0),
            'precio': item.get('precio', 0), 'total': pedido.get('total', 0), 'costo_envio': pedido.get('costoEnvio', 0),
            'metodo_pago': pedido.get('metodoPago', 'Sin definir'),
            'porcentaje_cancelados_previos': (historial.get(str(pedido.get('usuario', '')), (0, 0))[1] / historial.get(str(pedido.get('usuario', '')), (1, 0))[0] * 100) if historial.get(str(pedido.get('usuario', '')), (0, 0))[0] else 0
        })
df_pendientes = pd.DataFrame(filas_pendientes)
if df_pendientes.empty:
    print('No hay pedidos Pendiente/Pagado para clasificar.')
else:
    df_pendientes['riesgo'] = modelo.predict_proba(df_pendientes[features])[:, 1]
    resultado = df_pendientes.groupby('pedido_id', as_index=False).agg(usuario=('usuario', 'first'), riesgo=('riesgo', 'mean'))
    resultado['porcentaje'] = (resultado['riesgo'] * 100).round(1)
    resultado['nivel'] = pd.cut(resultado['riesgo'], [-1, .35, .60, 1], labels=['Bajo', 'Medio', 'Alto'])
    display(resultado.sort_values('riesgo', ascending=False))

In [ ]:
joblib.dump(modelo, 'modelo_riesgo_cancelacion.joblib')
print('Modelo guardado como modelo_riesgo_cancelacion.joblib')
client.close()